# Human Gene Essentiality Predictor V1.5
## Better Context-Dependent Prediction WITHOUT Expression Data

**Problem with V1:** Context-dependent genes had 43% accuracy (worse than random!)

**V1.5 Fix:** Use continuous Chronos scores as features, not just binary consensus

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Load CRISPR data
CRISPR_PATH = '/content/drive/MyDrive/depmap_data/CRISPRGeneEffect.csv'

print("Loading CRISPR data...")
crispr_raw = pd.read_csv(CRISPR_PATH, index_col=0).T  # genes x cell_lines
print(f"Shape: {crispr_raw.shape}")

# Binarize for labels
THRESHOLD = -0.5
binary_mat = (crispr_raw < THRESHOLD).astype(int)
print(f"Essential calls: {binary_mat.sum().sum():,} ({100*binary_mat.sum().sum()/binary_mat.size:.1f}%)")

In [ ]:
# Gene classification
gene_consensus = binary_mat.mean(axis=1)

HIGH_THRESH = 0.9
LOW_THRESH = 0.1

pan_mask = gene_consensus >= HIGH_THRESH
non_mask = gene_consensus <= LOW_THRESH
context_mask = ~pan_mask & ~non_mask

print(f"Pan-essential: {pan_mask.sum()}")
print(f"Non-essential: {non_mask.sum()}")
print(f"Context-dependent: {context_mask.sum()}")

## V1.5 Predictor: Use Chronos Scores as Features

In [ ]:
class HedgeFundV15:
    """
    V1.5: Use continuous Chronos scores for context-dependent genes.
    
    Key insight: The RAW Chronos score in the held-out cell line
    contains information we're not using!
    
    Wait... we can't use the held-out cell's score for prediction.
    
    NEW approach: For context-dependent genes, use CORRELATION patterns.
    If gene X is correlated with gene Y across cell lines, and we know
    Y's essentiality in the held-out line, we can predict X.
    """
    
    def __init__(self, high_thresh=0.9, low_thresh=0.1):
        self.high_thresh = high_thresh
        self.low_thresh = low_thresh
        self.ml_model = GradientBoostingClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.1
        )
        self.scaler = StandardScaler()
        
    def _get_features(self, crispr_raw, binary_mat, gene, cell_line, other_cells):
        """
        Features for a (gene, cell_line) pair:
        1. Binary consensus from other cell lines
        2. Mean Chronos score from other cell lines (continuous)
        3. Std of Chronos scores (variability)
        4. Percentile of this gene's mean score among all genes
        """
        # Binary consensus
        consensus = binary_mat.loc[gene, other_cells].mean()
        
        # Continuous Chronos features (from other cell lines)
        chronos_other = crispr_raw.loc[gene, other_cells]
        chronos_mean = chronos_other.mean()
        chronos_std = chronos_other.std()
        chronos_min = chronos_other.min()
        chronos_max = chronos_other.max()
        
        # How "essential-looking" is this gene overall?
        # (genes with very negative mean Chronos are more essential)
        
        return [
            consensus,
            chronos_mean,
            chronos_std,
            chronos_min,
            chronos_max,
            consensus ** 2,
            chronos_mean ** 2,
            consensus * abs(chronos_mean)  # interaction
        ]
    
    def fit(self, crispr_raw, binary_mat, n_train=50):
        """Train on a subset of cell lines."""
        print("Training V1.5 model...")
        
        gene_consensus = binary_mat.mean(axis=1)
        context_genes = gene_consensus[
            (gene_consensus > self.low_thresh) & 
            (gene_consensus < self.high_thresh)
        ].index.tolist()
        
        print(f"  Context-dependent genes: {len(context_genes)}")
        
        all_cells = list(binary_mat.columns)
        train_cells = all_cells[:n_train]
        
        X_train, y_train = [], []
        
        for cell in tqdm(train_cells, desc="  Building training data"):
            other = [c for c in all_cells if c != cell]
            for gene in context_genes:
                try:
                    feat = self._get_features(crispr_raw, binary_mat, gene, cell, other)
                    label = binary_mat.loc[gene, cell]
                    if not np.isnan(feat).any():
                        X_train.append(feat)
                        y_train.append(label)
                except:
                    pass
        
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        print(f"  Training samples: {len(X_train):,}")
        print(f"  Positive rate: {y_train.mean():.1%}")
        
        X_scaled = self.scaler.fit_transform(X_train)
        self.ml_model.fit(X_scaled, y_train)
        
        y_pred = self.ml_model.predict(X_scaled)
        print(f"  Train accuracy: {accuracy_score(y_train, y_pred):.3f}")
        print(f"  Train balanced: {balanced_accuracy_score(y_train, y_pred):.3f}")
        
        # Feature importance
        names = ['consensus', 'chronos_mean', 'chronos_std', 'chronos_min', 
                 'chronos_max', 'cons^2', 'chron^2', 'cons*chron']
        print("\n  Feature importance:")
        for n, imp in sorted(zip(names, self.ml_model.feature_importances_), key=lambda x: -x[1]):
            print(f"    {n:15s}: {imp:.3f}")
        
        self.is_fitted = True
        return self
    
    def predict(self, crispr_raw, binary_mat, cell_line):
        """Predict for one cell line."""
        all_cells = list(binary_mat.columns)
        other = [c for c in all_cells if c != cell_line]
        
        consensus = binary_mat[other].mean(axis=1)
        predictions = pd.Series(0, index=binary_mat.index)
        
        # Pan-essential
        predictions[consensus >= self.high_thresh] = 1
        
        # Context-dependent: ML
        context_mask = (consensus > self.low_thresh) & (consensus < self.high_thresh)
        context_genes = predictions[context_mask].index.tolist()
        
        if context_genes and self.is_fitted:
            X_test = []
            valid_genes = []
            for gene in context_genes:
                try:
                    feat = self._get_features(crispr_raw, binary_mat, gene, cell_line, other)
                    if not np.isnan(feat).any():
                        X_test.append(feat)
                        valid_genes.append(gene)
                except:
                    pass
            
            if X_test:
                X_test = self.scaler.transform(np.array(X_test))
                preds = self.ml_model.predict(X_test)
                for g, p in zip(valid_genes, preds):
                    predictions[g] = p
        
        return predictions

print("HedgeFundV15 defined!")

In [ ]:
# Train V1.5
predictor = HedgeFundV15()
predictor.fit(crispr_raw, binary_mat, n_train=50)

In [ ]:
# Evaluate on held-out cell lines
N_TEST = 50
all_cells = list(binary_mat.columns)
test_cells = all_cells[50:50+N_TEST]  # Skip training cells

all_y_true, all_y_pred = [], []

print(f"Evaluating on {len(test_cells)} held-out cell lines...")
for cell in tqdm(test_cells):
    y_true = binary_mat[cell].values
    y_pred = predictor.predict(crispr_raw, binary_mat, cell).values
    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)

y_true = np.array(all_y_true)
y_pred = np.array(all_y_pred)

print("\n" + "="*60)
print("V1.5 RESULTS")
print("="*60)
print(f"Accuracy:          {accuracy_score(y_true, y_pred):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.3f}")
print(f"F1 Score:          {f1_score(y_true, y_pred):.3f}")

In [ ]:
# Analyze by category
categories = {
    'Pan-essential (≥90%)': gene_consensus >= 0.9,
    'Common (50-90%)': (gene_consensus >= 0.5) & (gene_consensus < 0.9),
    'Context-dep (10-50%)': (gene_consensus >= 0.1) & (gene_consensus < 0.5),
    'Rarely ess (0-10%)': (gene_consensus > 0) & (gene_consensus < 0.1),
    'Non-essential (0%)': gene_consensus == 0
}

# V1 baseline (from your results)
v1_results = {
    'Pan-essential (≥90%)': 0.982,
    'Common (50-90%)': 0.434,
    'Context-dep (10-50%)': 0.439,
    'Rarely ess (0-10%)': 0.991,
    'Non-essential (0%)': 1.000
}

print("\nAccuracy by Category (V1 → V1.5):")
print("-" * 70)

for cat_name, mask in categories.items():
    n_genes = mask.sum()
    if n_genes > 0:
        mask_exp = np.tile(mask.values, N_TEST)
        if mask_exp.sum() > 0:
            acc_v15 = accuracy_score(y_true[mask_exp], y_pred[mask_exp])
            acc_v1 = v1_results.get(cat_name, 0)
            diff = acc_v15 - acc_v1
            emoji = "✅" if diff > 0.05 else "➡️" if diff > -0.05 else "❌"
            print(f"{cat_name:25s}: {acc_v1:.1%} → {acc_v15:.1%} ({diff:+.1%}) {emoji}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(12, 6))

cats = list(v1_results.keys())
v1_accs = list(v1_results.values())

# Calculate V1.5 accuracies
v15_accs = []
for cat_name, mask in categories.items():
    mask_exp = np.tile(mask.values, N_TEST)
    if mask_exp.sum() > 0:
        v15_accs.append(accuracy_score(y_true[mask_exp], y_pred[mask_exp]))
    else:
        v15_accs.append(0)

x = np.arange(len(cats))
width = 0.35

bars1 = ax.bar(x - width/2, v1_accs, width, label='V1 (Consensus)', color='#ff6b6b', alpha=0.8)
bars2 = ax.bar(x + width/2, v15_accs, width, label='V1.5 (+ Chronos)', color='#4ecdc4', alpha=0.8)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Accuracy')
ax.set_title('V1 vs V1.5: Prediction Accuracy by Gene Category')
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

for bar, acc in zip(bars1, v1_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{acc:.1%}', ha='center', fontsize=9)
for bar, acc in zip(bars2, v15_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{acc:.1%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Summary

V1.5 uses continuous Chronos scores as additional features for context-dependent genes.

If this doesn't improve context-dependent accuracy significantly, the next step is to download expression data for V2.